# Arriendo de viviendas en Madrid



Importación de las librerías

In [1]:
# Para la manipulación de datos
import pandas as pd
import numpy as np

# Servicio y driver de Chrome de Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Las opciones que vamos a tener para buscar elementos
from selenium.webdriver.common.by import By

# Para cuando queramos mandar pulsaciones de teclado
from selenium.webdriver.common.keys import Keys

# Hacemos que espere
import time

# Importaciones para esperas explícitas (mejor práctica que time.sleep)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Importamos undetected-chromedriver para evitar el captcha
import undetected_chromedriver as uc

# Importamos excepciones comunes de Selenium para manejo de errores
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

In [2]:
service = Service(ChromeDriverManager().install())

In [4]:
from selenium import webdriver

driver = webdriver.Chrome()
print(driver.capabilities["browserVersion"])
driver.quit()

146.0.7680.178


Creamos el driver para controlar el navegador

In [5]:
# Aquí empiezo ejecutando el robot

import undetected_chromedriver as uc

driver = uc.Chrome(
    version_main=146,  #Para que funcione con chrome 146
    browser_executable_path=r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    service=service,
    use_subprocess=False,
    headless=False,
)

Accedemos a la página principal

In [6]:
url = "https://www.idealista.com/areas/alquiler-viviendas/con-metros-cuadrados-mas-de_40,de-dos-dormitorios,de-tres-dormitorios,de-cuatro-cinco-habitaciones-o-mas,alquiler-de-larga-temporada/?shape=%28%28siyuFfguUcyDa%5DkSa%7B%40uM%7Bw%40fBkp%40zmDnY%7Cw%40J%7BLnhD%29%29"
driver.get(url)

Hay un botón de Consentir datos personales

In [6]:
wait = WebDriverWait(driver, 10)

accept = wait.until(EC.element_to_be_clickable((
    By.XPATH, "//p[contains(@class, 'fc-button-label') and contains(text(), 'Consentir')]"
)))
accept.click()

Hago una funcion para guardar los datos más importantes de cada propiedad: título, 

In [9]:

URL_BASE = "https://www.idealista.com/areas/alquiler-viviendas/con-metros-cuadrados-mas-de_40,de-dos-dormitorios,de-tres-dormitorios,de-cuatro-cinco-habitaciones-o-mas,alquiler-de-larga-temporada/?shape=%28%28siyuFfguUcyDa%5DkSa%7B%40uM%7Bw%40fBkp%40zmDnY%7Cw%40J%7BLnhD%29%29&pagina={}"
TOTAL_PAGINAS =21  # ajusta al total real

todos_los_datos = []

for pagina in range(1, TOTAL_PAGINAS + 1):
    driver.get(URL_BASE.format(pagina))
    
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.item")))
    time.sleep(3)  # pausa extra para evitar bloqueos

    publicaciones = driver.find_elements(By.CSS_SELECTOR, "article.item")

    for pub in publicaciones:
        try:
            titulo_el = pub.find_element(By.CSS_SELECTOR, "a.item-link")
            titulo = titulo_el.get_attribute("title").strip()
        except:
            titulo = "N/A"
        try:
            precio = pub.find_element(By.CSS_SELECTOR, "span.item-price").text.strip()
        except:
            precio = "N/A"

        # habitaciones, m2 y altura están todos en span.item-detail
        habitaciones = "N/A"
        m2 = "N/A"
        altura = "N/A"
        try:
            detalles = pub.find_elements(By.CSS_SELECTOR, "div.item-detail-char span.item-detail")
            for detalle in detalles:
                texto = detalle.text.strip()
                if "hab." in texto:
                    habitaciones = texto.replace("hab.", "").strip()
                elif "m²" in texto:
                    m2 = texto.replace("m²", "").strip()
                elif "Planta" in texto or "Bajo" in texto or "Entresuelo" in texto:
                    altura = texto
        except:
            pass

        todos_los_datos.append({
            "titulo": titulo,
            "precio": precio,
            "habitaciones": habitaciones,
            "m2": m2,
            "altura": altura
        })

    print(f"Página {pagina}/{TOTAL_PAGINAS} — total registros: {len(todos_los_datos)}")
    time.sleep(3)  # pausa entre páginas

print(f"✅ Total propiedades extraídas: {len(todos_los_datos)}")

Página 1/21 — total registros: 30
Página 2/21 — total registros: 60
Página 3/21 — total registros: 90
Página 4/21 — total registros: 120
Página 5/21 — total registros: 150
Página 6/21 — total registros: 180
Página 7/21 — total registros: 210
Página 8/21 — total registros: 240
Página 9/21 — total registros: 270
Página 10/21 — total registros: 300
Página 11/21 — total registros: 330
Página 12/21 — total registros: 360
Página 13/21 — total registros: 390
Página 14/21 — total registros: 420
Página 15/21 — total registros: 450
Página 16/21 — total registros: 480
Página 17/21 — total registros: 510
Página 18/21 — total registros: 540
Página 19/21 — total registros: 570
Página 20/21 — total registros: 600
Página 21/21 — total registros: 630
✅ Total propiedades extraídas: 630


In [10]:
df_Pisos = pd.DataFrame(todos_los_datos)
df_Pisos.head()

,titulo,precio,habitaciones,m2,altura
0,"Piso en Calle de las Almortas, Valdeacederas, ...",1.590€/mes,2,74,Planta 2ª exterior sin ascensor
1,"Piso en Calle Esquilache, 14, Vallehermoso, Ma...",1.500€/mes,2,100,Bajo exterior con ascensor
2,"Piso en Calle de Olite, 48, Bellas Vistas, Madrid",1.700€/mes,2,68,Planta 4ª exterior con ascensor
3,"Piso en Paseo de La Habana, El Viso, Madrid",2.950€/mes,2,134,Planta 2ª exterior con ascensor
4,Piso en Calle de Raimundo Fernández Villaverde...,1.450€/mes,2,43,Planta 1ª interior con ascensor


In [14]:
import re
def extraer_zona(titulo):
    partes = titulo.split(',')
    # filtrar partes que sean solo números o espacios
    partes_limpias = [p.strip() for p in partes if not re.match(r'^\s*\d+\s*$', p)]
    # descartar la primera parte (tipo + calle) y tomar el resto
    return ', '.join(partes_limpias[1:]).strip()

df_Pisos['zona'] = df_Pisos['titulo'].apply(extraer_zona)
df_Pisos.head(10)

,titulo,precio,habitaciones,m2,altura,zona
0,"Piso en Calle de las Almortas, Valdeacederas, ...",1.590€/mes,2,74,Planta 2ª exterior sin ascensor,"Valdeacederas, Madrid"
1,"Piso en Calle Esquilache, 14, Vallehermoso, Ma...",1.500€/mes,2,100,Bajo exterior con ascensor,"Vallehermoso, Madrid"
2,"Piso en Calle de Olite, 48, Bellas Vistas, Madrid",1.700€/mes,2,68,Planta 4ª exterior con ascensor,"Bellas Vistas, Madrid"
3,"Piso en Paseo de La Habana, El Viso, Madrid",2.950€/mes,2,134,Planta 2ª exterior con ascensor,"El Viso, Madrid"
4,Piso en Calle de Raimundo Fernández Villaverde...,1.450€/mes,2,43,Planta 1ª interior con ascensor,"Cuatro Caminos, Madrid"
5,"Ático en Calle Santa Engracia, Nuevos Minister...",3.000€/mes,3,108,Planta 7ª exterior con ascensor,"Nuevos Ministerios-Ríos Rosas, Madrid"
6,"Piso en Valdezarza, Madrid",1.400€/mes,3,85,Planta 3ª exterior sin ascensor,Madrid
7,"Piso en Plaza Diego de Ordas, 2, Nuevos Minist...",2.150€/mes,2,86,Planta 3ª exterior con ascensor,"Nuevos Ministerios-Ríos Rosas, Madrid"
8,"Piso en Calle Cantueso, 8, Valdeacederas, Madrid",1.600€/mes,2,85,Planta 2ª exterior con ascensor,"Valdeacederas, Madrid"
9,"Piso en Paseo de la Dirección, 161, Valdeacede...",2.950€/mes,3,145,Planta 17ª exterior con ascensor,"Valdeacederas, Madrid"


In [15]:
df_Pisos.to_csv(r"C:\Users\Ariela\Desktop\EDA - Inmobiliario\EDA_Inmobiliario_Madrid", index=False, encoding="utf-8-sig")